<a href="https://colab.research.google.com/github/alheliou/Bias_mitigation/blob/main/UPP26/TD5_inprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TD 5: Mitigation des biais avec une méthode de in-processsing Prejudice Remover

The aim of this notebook is to use the Prejudice Remover in-processing approach and analyse its impact on the model output.
In terms of Machine Learning we will go a bit further in the train/valid/test paradigm.

The model has to be learn on the train dataset, then the model parameters has to be optimized on the valid dataset, and finally the model performance is evaluated on the test dataset.
No choice/decision etc can be taken depending on the test dataset. This could result on an overfitting on the test dataset.

Here you will manipulate:
- Prejudice Remover approach as a black box
- Training of the prejudice remover using the train/valid paradigm. to choice the 'best' threshold
- Combine Prejudice Remover with Reweighing

As a reminder of pre-processing approach we encourage you to :
- analyse the impact of the Reweighing on different model (Logistic Regression, Decision Tree, Random Forest, etc.)


## Installation of the environnement

We highly recommend you to follow these steps, it will allow every student to work in an environment as similar as possible to the one used during testing.

### Colab Settings ---- for Colab Users ONLY
  The next cell of code are to execute only once per colab environment


#### Python env creation (Colab only)

        ```
        ! python -m pip install numpy fairlearn plotly nbformat ipykernel aif360["inFairness"] aif360['AdversarialDebiasing'] causal-learn BlackBoxAuditing cvxpy dice-ml lime shapkit
        ```
#### 2. Download MEPS dataset (for part2) it can take several minutes (Colab only)

        ```
        ! Rscript /usr/local/lib/python3.12/dist-packages/aif360/data/raw/meps/generate_data.R
        ! mv h181.csv /usr/local/lib/python3.12/dist-packages/aif360/data/raw/meps/
        ! mv h192.csv /usr/local/lib/python3.12/dist-packages/aif360/data/raw/meps/
        ```

### Local Settings ---- for installation on local computer ONLY

If you arleady have an env from TD2, TD3 or TD4, you can simply reuse it.


#### 1. Uv installation (local only, no need to redo if already done)


        https://docs.astral.sh/uv/getting-started/installation/


        `curl -LsSf https://astral.sh/uv/install.sh | sh`

        Python version 3.12 installation (highly recommended)
        `uv python install 3.12`

#### 2. R installation *NEW* (local only)

        In the command `Rscript` says 'command not found'

        `sudo apt install r-base-core`

#### 3. Python env creation (local only, no need to redo if already done)

        ```
        mkdir TD_bias_mitigation
        cd TD_bias_mitigation
        uv python pin 3.12
        uv init
        uv venv
        uv add numpy fairlearn plotly nbformat ipykernel aif360["inFairness"] aif360['AdversarialDebiasing'] causal-learn BlackBoxAuditing cvxpy dice-ml lime shapkit
        uv add pandas==2.2.2
        ```

#### 4. Download MEPS dataset, it can take several minutes *NEW* (local only)

        ```
        cd TD_bias_mitigation/.venv/lib/python3.12/site-packages/aif360/data/raw/meps/
        Rscript generate_data.R
        ```

In [ ]:
# To execute only in Colab
! python -m pip install numpy fairlearn plotly nbformat ipykernel aif360["inFairness"] aif360['AdversarialDebiasing'] causal-learn BlackBoxAuditing cvxpy dice-ml lime shapkit

In [ ]:
# To execute only in Colab
! Rscript /usr/local/lib/python3.12/dist-packages/aif360/data/raw/meps/generate_data.R
! mv h181.csv /usr/local/lib/python3.12/dist-packages/aif360/data/raw/meps/
! mv h192.csv /usr/local/lib/python3.12/dist-packages/aif360/data/raw/meps/

## 1. Import and load the dataset

In [1]:
# imports
import numpy as np
import pandas as pd
import plotly.express as px
import warnings

warnings.simplefilter(action="ignore", category=FutureWarning)
warnings.simplefilter(action="ignore", append=True, category=UserWarning)
# Datasets
from aif360.datasets import MEPSDataset19

# Fairness metrics
from sklearn.metrics import accuracy_score, balanced_accuracy_score
from sklearn.preprocessing import StandardScaler

MEPSDataset19_data = MEPSDataset19()
(dataset_orig_panel19_train, dataset_orig_panel19_val, dataset_orig_panel19_test) = (
    MEPSDataset19().split([0.5, 0.8], shuffle=True)
)

In [2]:
len(dataset_orig_panel19_train.instance_weights), len(
    dataset_orig_panel19_val.instance_weights
), len(dataset_orig_panel19_test.instance_weights)

(7915, 4749, 3166)

In [3]:
instance_weights = MEPSDataset19_data.instance_weights
instance_weights

array([21854.981705, 18169.604822, 17191.832515, ...,  3896.116219,
        4883.851005,  6630.588948], shape=(15830,))

In [4]:
f"Taille du dataset {len(instance_weights)}, poids total du dataset {instance_weights.sum()}."

'Taille du dataset 15830, poids total du dataset 141367240.546316.'

In [5]:
from aif360.sklearn.metrics import *
from sklearn.metrics import  balanced_accuracy_score


# This method takes lists
def get_metrics(
    y_true, # list or np.array of truth values
    y_pred=None,  # list or np.array of predictions
    prot_attr=None, # list or np.array of protected/sensitive attribute values
    priv_group=1, # value taken by the privileged group
    pos_label=1, # value taken by the positive truth/prediction
    sample_weight=None # list or np.array of weights value,
):
    group_metrics = {}
    group_metrics["base_rate_truth"] = base_rate(
        y_true=y_true, pos_label=pos_label, sample_weight=sample_weight
    )
    group_metrics["statistical_parity_difference"] = statistical_parity_difference(
        y_true=y_true, y_pred=y_pred, prot_attr=prot_attr, priv_group=priv_group, pos_label=pos_label, sample_weight=sample_weight
    )
    group_metrics["disparate_impact_ratio"] = disparate_impact_ratio(
        y_true=y_true, y_pred=y_pred, prot_attr=prot_attr, priv_group=priv_group, pos_label=pos_label, sample_weight=sample_weight
    )
    if not y_pred is None:
        group_metrics["base_rate_preds"] = base_rate(
        y_true=y_pred, pos_label=pos_label, sample_weight=sample_weight
        )
        group_metrics["equal_opportunity_difference"] = equal_opportunity_difference(
            y_true=y_true, y_pred=y_pred, prot_attr=prot_attr, priv_group=priv_group, pos_label=pos_label, sample_weight=sample_weight
        )
        group_metrics["average_odds_difference"] = average_odds_difference(
            y_true=y_true, y_pred=y_pred, prot_attr=prot_attr, priv_group=priv_group, pos_label=pos_label, sample_weight=sample_weight
        )
        if len(set(y_pred))>1:
            group_metrics["conditional_demographic_disparity"] = conditional_demographic_disparity(
                y_true=y_true, y_pred=y_pred, prot_attr=prot_attr, pos_label=pos_label, sample_weight=sample_weight
            )
        else:
            group_metrics["conditional_demographic_disparity"] =None
        group_metrics["smoothed_edf"] = smoothed_edf(
        y_true=y_true, y_pred=y_pred, prot_attr=prot_attr, pos_label=pos_label, sample_weight=sample_weight
        )
        group_metrics["df_bias_amplification"] = df_bias_amplification(
        y_true=y_true, y_pred=y_pred, prot_attr=prot_attr, pos_label=pos_label, sample_weight=sample_weight
        )
        group_metrics["balanced_accuracy_score"] = balanced_accuracy_score(
        y_true=y_true, y_pred=y_pred, sample_weight=sample_weight
        )
    return group_metrics

2026-02-20 10:09:39.804975: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-20 10:09:40.473907: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-20 10:09:42.997197: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


## Learning a Prejudice Remover model on the training dataset, and choose the best parameters with the validation dataset

In [ ]:
# Bias mitigation techniques
from aif360.algorithms.preprocessing import Reweighing
from aif360.algorithms.inprocessing import PrejudiceRemover

### Question1 : Learn a Standard Scaler on the training dataset features, its output will be used as input of the model learned

In [ ]:
from sklearn.preprocessing import StandardScaler
dataset_train, dataset_val, dataset_test = MEPSDataset19().split([0.5, 0.8], shuffle=True)

# Learn scaler on TRAIN features only
scaler = StandardScaler()
X_train = dataset_train.features
scaler.fit(X_train)

# Transform train/val/test with the SAME scaler
dataset_train_s = dataset_train.copy()
dataset_val_s   = dataset_val.copy()
dataset_test_s  = dataset_test.copy()

dataset_train_s.features = scaler.transform(dataset_train.features)
dataset_val_s.features   = scaler.transform(dataset_val.features)
dataset_test_s.features  = scaler.transform(dataset_test.features)

print(dataset_train_s.features.mean(axis=0)[:5])
print(dataset_train_s.features.std(axis=0)[:5])

[ 2.35089550e-17 -1.14543035e-16  1.17783126e-14 -5.67698482e-15
  2.46170740e-17]
[1. 1. 1. 1. 1.]


### Question2: Create a method to learn a Prejudice Remover on the train dataset and retrieve the model learned
Execute the method with the parameter eta arbitrarily set at 25.0



In [9]:
from aif360.algorithms.inprocessing import PrejudiceRemover

def learn_prejudice_remover(train_dataset, *, eta=25.0, sensitive_attr=None):
    """
    Entraîne PrejudiceRemover sur le dataset d'entraînement et renvoie :
    - le modèle (objet PrejudiceRemover fitted)
    - les prédictions sur le train (BinaryLabelDataset) si tu veux vérifier direct
    """
    # Si tu ne précises pas l'attribut sensible, on prend le 1er du dataset
    if sensitive_attr is None:
        if not train_dataset.protected_attribute_names:
            raise ValueError("Le dataset n'a pas de protected_attribute_names.")
        sensitive_attr = train_dataset.protected_attribute_names[0]

    model = PrejudiceRemover(sensitive_attr=sensitive_attr, eta=eta)
    model.fit(train_dataset)

    return model  # <- modèle appris (fitted)

In [12]:
pr_model = learn_prejudice_remover(dataset_train_s, eta=25.0)

print("Sensitive attribute used:", pr_model.sensitive_attr)
print("Eta:", pr_model.eta)

/home/brion/Fairness/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/brion/Fairness/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


Sensitive attribute used: RACE
Eta: 25.0


Le score du Prejudice Remover donne un sortie pour chaque instance une seule valeur, c'est un seuil, arbritrairement fixé à 0.5 par défault, qui permet à partir de ce score de décider la prédiction 1 ou 0.
Si le score est supérieur au seuil la prédiction est 1, sinon c'est 0.

### Validating: Choose the best parameters

Here there are two parameters :
- eta: fairness penalty parameter of the PR model
- thershold: the threshold of the binary classification

The threshold is used to obtains predictions from the model output.
The eta is used during the training

Question3: Create a method that will loop over 50 threshold ]0:0.5( and 5 values of ETA [1.0: 100.0], and outputs the metrics

In [13]:
import numpy as np
import pandas as pd

from sklearn.metrics import accuracy_score, balanced_accuracy_score
from aif360.algorithms.inprocessing import PrejudiceRemover
from aif360.metrics import ClassificationMetric

def grid_search_pr_threshold_eta(
    dataset_train,
    dataset_val,
    privileged_groups,
    unprivileged_groups,
    *,
    sensitive_attr=None,
    etas=None,
    thresholds=None,
):
    """
    Loop over eta (training) and threshold (decision) and output metrics.

    Parameters
    ----------
    dataset_train, dataset_val: AIF360 BinaryLabelDataset
    privileged_groups, unprivileged_groups: list[dict] for AIF360 metrics
        e.g. privileged_groups=[{"RACE": 1}], unprivileged_groups=[{"RACE": 0}]
    sensitive_attr: str or None
        If None, uses the first protected attribute name in dataset_train
    etas: iterable[float] or None
        If None -> 5 log-spaced values in [1, 100]
    thresholds: iterable[float] or None
        If None -> 50 values in ]0, 0.5[
    """

    # defaults
    if sensitive_attr is None:
        if not dataset_train.protected_attribute_names:
            raise ValueError("No protected_attribute_names found in dataset_train.")
        sensitive_attr = dataset_train.protected_attribute_names[0]

    if etas is None:
        etas = np.logspace(0, 2, 5)  # 1.0 .. 100.0 (5 values)

    if thresholds is None:
        # 50 thresholds in ]0, 0.5[
        thresholds = np.linspace(1e-6, 0.5 - 1e-6, 50)

    # convenience
    fav = dataset_train.favorable_label
    unfav = dataset_train.unfavorable_label

    rows = []

    # ground truth for perf metrics (convert to 1D)
    y_true = dataset_val.labels.ravel()

    for eta in etas:
        # 1) fit model on TRAIN
        model = PrejudiceRemover(sensitive_attr=sensitive_attr, eta=float(eta))
        model.fit(dataset_train)

        # 2) predict on VAL (gives scores)
        pred = model.predict(dataset_val)

        # scores may be in pred.scores (typical AIF360); fall back to labels if needed
        if hasattr(pred, "scores") and pred.scores is not None:
            scores = pred.scores.ravel()
        else:
            # Some implementations may already output probabilities/labels in labels
            scores = pred.labels.ravel().astype(float)

        for thr in thresholds:
            # 3) threshold -> binary labels with correct favorable/unfavorable labels
            pred_thr = pred.copy()
            pred_labels = np.where(scores >= thr, fav, unfav).reshape(-1, 1)
            pred_thr.labels = pred_labels

            y_pred = pred_thr.labels.ravel()

            # 4) performance metrics
            acc = accuracy_score(y_true, y_pred)
            bacc = balanced_accuracy_score(y_true, y_pred)

            # 5) fairness metrics (AIF360)
            cm = ClassificationMetric(
                dataset_val,
                pred_thr,
                unprivileged_groups=unprivileged_groups,
                privileged_groups=privileged_groups,
            )

            rows.append({
                "eta": float(eta),
                "threshold": float(thr),
                "accuracy": acc,
                "balanced_accuracy": bacc,
                "statistical_parity_difference": cm.statistical_parity_difference(),
                "disparate_impact": cm.disparate_impact(),
                "equal_opportunity_difference": cm.equal_opportunity_difference(),
                "average_odds_difference": cm.average_odds_difference(),
                "theil_index": cm.theil_index(),
            })

    return pd.DataFrame(rows)

In [ ]:
privileged_groups = [{"RACE": 1}]
unprivileged_groups = [{"RACE": 0}]

results = grid_search_pr_threshold_eta(
    dataset_train_s,
    dataset_val_s,
    privileged_groups,
    unprivileged_groups,
    sensitive_attr="RACE",
)

results.head()

/home/brion/Fairness/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/brion/Fairness/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/brion/Fairness/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. 

,eta,threshold,accuracy,balanced_accuracy,statistical_parity_difference,disparate_impact,equal_opportunity_difference,average_odds_difference,theil_index
0,1.0,0.000001,0.168246,0.500000,0.000000,1.000000,0.000000,0.000000,0.024208
1,1.0,0.010205,0.212676,0.524712,-0.053275,0.946220,-0.002062,-0.030850,0.030814
2,1.0,0.020409,0.291430,0.567561,-0.143880,0.851859,-0.003989,-0.082162,0.042090
3,1.0,0.030613,0.374395,0.612942,-0.224852,0.759925,-0.017790,-0.131156,0.052428
4,1.0,0.040817,0.451885,0.652535,-0.269306,0.696824,-0.037838,-0.161712,0.061624


### Question4 : Make plot to choose the best set of parameters

In [ ]:
print("TODO")

### Question 5: Evaluate : compute the metrics on the test dataset using the model learnt with the selected parameters


In [ ]:
print("TODO")

## Combine pre-processing and in-processing
### Question6: Redo the Prejudice Remover approach using first the Reweighing pre-processing

In [ ]:
print("TODO")

## Adversarial Debiasing

Adversarial debiasing [1] is an in-processing technique that learns a classifier to maximize prediction accuracy and simultaneously reduce an adversary's ability to determine the protected attribute from the predictions.

See [AIF360 tuto](https://github.com/Trusted-AI/AIF360/blob/main/examples/demo_adversarial_debiasing.ipynb)

Here we show how to learn and Adversarial Debiasing with the argumetn debias set to False

In [ ]:
import tensorflow.compat.v1 as tf
tf.disable_eager_execution()
from aif360.algorithms.inprocessing.adversarial_debiasing import AdversarialDebiasing

sess = tf.Session()

plain_model = AdversarialDebiasing(
    unprivileged_groups=[{'RACE': 0.0}],
    privileged_groups=[{'RACE': 1.0}],
    scope_name='plain_classifier',
    debias=False,
    sess=sess)

plain_model.fit(dataset_orig_panel19_train)

In [ ]:
# Apply the plain model to train and val data
dataset_nodebiasing_train = plain_model.predict(dataset_orig_panel19_train)
dataset_nodebiasing_val = plain_model.predict(dataset_orig_panel19_val)

In [ ]:
get_metrics(
    y_true = dataset_orig_panel19_train.labels[:,0],
    y_pred= dataset_nodebiasing_train.labels[:,0],
    prot_attr= dataset_orig_panel19_train.protected_attributes[:,0],
    sample_weight= dataset_orig_panel19_train.instance_weights
)

In [ ]:
get_metrics(
    y_true = dataset_orig_panel19_val.labels[:,0],
    y_pred= dataset_nodebiasing_val.labels[:,0],
    prot_attr= dataset_orig_panel19_val.protected_attributes[:,0],
    sample_weight= dataset_orig_panel19_val.instance_weights
)

In [ ]:
sess.close()
tf.reset_default_graph()


### Question 7: Redo the same (learn and Adversarial Debiasing) with the argument debias set to True

Compare the metrics outputed

In [ ]:
print("TODO")

### Question 8: Combine the Reweighing with the Adversarial Debiasing

In [ ]:
print("TODO")

This in-processing approach does not seem compatible withe the Reweighing, has the df_bias_amplification is high and the disparate impact ratio is not improved by the use of the reweighing has pre-processing.
Although very efficient on the fairness metrics of the dataset, the Reweighing is not convenient for every kind of machine learning algo.



## Analysis of the influence of Reweighing

### QUESTION 9 : Pour aller plus loin, étudier l'impact du Reweighing sur différents modèles notamment les arbres de décision

In [ ]:
print("TODO")